# 03 · Skewers

`SkewerBundle` places the sampling points of $N_{\rm sk}$ skewers, sets the photon directions and solves $a(x)$ along
each one with a `Cosmology` (02). Every field is sampled at its points and line elements (04), and the simulation gas
is deposited on its dual segments (05). Runtime: ~10 s.

**Requires:** a cosmology (02) to solve $a(x)$, or `manual_scale_factors`.  
**Independent of:** lines, fields.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('cosmology', 'skewers',)); plt.show()

In [ ]:
from skewerjax.core import Cosmology, SkewerBundle, phys, path_geometry, plane_points, grid_skewer_points

## 1 · SkewerBundle: points, photon direction and $a(x)$

Each skewer has its initial condition ($a_0$ at comoving distance $x_0$) at its first point (`ICs_defined_at_index=0`)
or at its last one (`-1`). That point is either where the light is emitted or where it is absorbed
(`ICs_at_emission_or_absorption`). These two choices fix the photon direction relative to the order of the points
(`array_direction`, $\pm 1$) and how $a(x)$ is integrated.

In [ ]:
L = 1000.0
px = L * jnp.array([[0.0, 0.5, 0.5], [0.2, 0.5, 0.5], [0.3, 0.5, 0.5], [0.5, 0.5, 0.5], [1.0, 0.5, 0.5]])
planck = Cosmology(H0=phys.H0, Omega_r_0=9.1436e-5, Omega_m_0=0.3089, Omega_k_0=0.0, Omega_lambda_0=0.6911)
configs = [(0, "emission"), (0, "absorption"), (-1, "emission"), (-1, "absorption")]
sb = SkewerBundle(spatial_points=jnp.stack([px] * 4), a0=1.0, x0=0.0, ICs_defined_at_index=jnp.array([c[0] for c in configs]),
                  ICs_at_emission_or_absorption=[c[1] for c in configs], cosmologies=planck)
print("array_direction:", np.asarray(sb.array_direction), "| quadri_vectors (N_cosmo, N_sk, N_ppsk, 4):", sb.quadri_vectors.shape)
fig, ax = plt.subplots(figsize=(13, 6), layout="constrained")
for k, (i0, kind) in enumerate(configs):
    ax.plot(np.asarray(px[:, 0]), np.asarray(sb.scale_factors[0, k]), "o-", color=list(TOL.values())[k],
            label=f"ICs at point {i0} = {kind} (direction {int(sb.array_direction[k]):+d})")
ax.set(xlabel="x [Mpc]", ylabel="a(x)", title="The same points, four conventions: a = 1 where the ICs are set"); ax.legend(); plt.show()

## 2 · One skewer: its geometry and what the bundle stores

`path_geometry(points)` returns, for any polyline:

- the path length $s$;
- the line element $w_i$ of every point;
- the unit tangents;
- the midpoints $m_{i\pm 1/2}$ that bound the *dual segment* of every point.

`SkewerBundle` exposes them as `path_lengths`, `line_elements`, `tangents` and `segment_midpoints`. The figure below
shows them on one helical skewer, sampled unevenly so that the line elements differ from point to point:

- **Sampling points** $x_i$ (dots): where every field is evaluated (`Fields.sample`) and where $a(x)$ is solved.
- **Midpoints** $m_{i\pm 1/2}$ (diamonds): halfway to each neighbour. They bound the **dual segment** of point $i$, the
  stretch of path that point $i$ stands for (the thick bands, coloured by $a(x)$). The two end points own half a segment.
- **Line elements** $w_i$: the length of the dual segment,

$$
w_i = \tfrac12\left(\lvert x_{i+1} - x_i\rvert + \lvert x_i - x_{i-1}\rvert\right),\qquad \sum_i w_i = \text{path length}.
$$

  They are the quadrature weights of the optical depth (06), and they tile the path.
- **Unit tangents** $\hat t_i$ (arrows): the direction of the path at each point (central difference). The
  line-of-sight peculiar velocity is $v_\parallel = \mathbf{v}\cdot\hat t_i$. In a simulation, the deposition (05)
  cuts every dual segment into the pieces of the cells it crosses.
- **The colour** is the scale factor that `SkewerBundle` solves along the path (Planck 2018, the observer at the last
  point, at $z = 2$). It changes along the line of sight only, because $a(x)$ is a function of the comoving distance
  travelled by the light.

In [ ]:
from skewerjax.utils.plotting import plot_skewer_geometry_3d, plot_skewer_anatomy
Lb = 10.0
tu = jnp.linspace(0.0, 1.0, 18) ** 1.6                                      # uneven sampling: the line elements vary
helix = jnp.stack([Lb / 2 + 0.35 * Lb * jnp.cos(2 * jnp.pi * tu), Lb / 2 + 0.35 * Lb * jnp.sin(2 * jnp.pi * tu), Lb * tu], -1)
sb_h = SkewerBundle(spatial_points=helix[None], a0=1 / 3, x0=0.0, ICs_defined_at_index=-1, ICs_at_emission_or_absorption="absorption",
                    cosmologies=planck)
g = path_geometry(helix)
fig = plt.figure(figsize=(18, 12), layout="constrained")
ax = fig.add_subplot(projection="3d")
sm = plot_skewer_geometry_3d(ax, helix, scale_factors=sb_h.scale_factors[0, 0], highlight=7, label_w=(0, 3, 11, 14, 17),
                             tangent_length=0.6)
ax.view_init(24, -58)
ax.legend(loc="upper left", fontsize=15)
fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.02, label="scale factor a(x) (observer at the last point)")
ax.set_title("One skewer: points, midpoints, dual segments, tangents and line elements", fontsize=20)
plt.show()
print(f"sum of the line elements {float(g['line_elements'].sum()):.6f} = path length {float(g['s'][-1]):.6f} cMpc; "
      f"w_i from {float(g['line_elements'][1:-1].min()):.3f} to {float(g['line_elements'].max()):.3f} cMpc")

Everything the bundle stores for that same skewer:

- (a) the positions;
- (b) the two distances: the path length from the first point, and the comoving distance from the initial condition;
- (c) the line elements of the figure above;
- (d) $a(x)$ with its initial condition;
- (e) the photon directions, along the tangents here, because the light travels towards the last point, where it is
  absorbed.

Every array is a pytree leaf. `quadri_vectors` is the batch of events $(a, x, y, z)$ that the field samplers (04)
take.

In [ ]:
fig, axs = plot_skewer_anatomy(sb_h, 0)
fig.suptitle("What a SkewerBundle stores for the skewer above: positions, distances, line elements, a(x), photon directions", fontsize=21)
plt.show()

**Options** of `SkewerBundle`:

- `a0`, `x0`, `ICs_defined_at_index` and `ICs_at_emission_or_absorption`: scalars or one per skewer.
- `los_directions`: otherwise derived from the points.
- `manual_scale_factors`: otherwise solved with `cosmologies`, which may hold several backgrounds (one set of
  quadri-vectors per cosmology).

Point layouts for many skewers: `plane_points`, `grid_skewer_points`, `axis_aligned_skewer_points`. The bundle is a
pytree.